# BlinkMimic-T4 训练后评测 + 视频

对最新完成的 run `20261122_105913` 做三件事：

1. `tools/eval_metrics.py`：归一化回报、失败动作数、身体位置/姿态误差、根漂移、关节误差、脚滑、接触一致率、穿地
2. `blinkmimic.evaluate --record`：录制 40 段动作的确定性 rollout
3. `tools/render.py`：渲染走、跑、转身、舞蹈四段视频

结果写入 run 目录，并上传到 Hugging Face 私有仓库；视频直接在 Notebook 里显示。

只做仿真评测，不涉及真机。


In [ ]:
from pathlib import Path
import os
import shutil

RUN_ID = '20261122_105913'
REPO_DIR = Path('/kaggle/working/proj2-humanoid-mimic/third_party/blinkmimic')
REPO_URL = 'https://github.com/fuyichen0725/blinkmimic.git'
COMMIT = 'd264658d6c9ce61fb52f9f0f8985a369f3ccc409'
BASE_RESULTS = Path('/kaggle/working/proj2-humanoid-mimic/results/blinkmimic_t4_ddp')
LOG_ROOT = BASE_RESULTS / f'run_{RUN_ID}'
HF_REPO_NAME = 'proj2-blinkmimic-t4'

VIDEO_SECONDS = 10
VIDEO_WIDTH = 640
VIDEO_HEIGHT = 360
VIDEO_FPS = 15

HF_TOKEN = os.environ.get('HF_TOKEN', '')
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN') or HF_TOKEN
except Exception as exc:
    print(f'kaggle_secrets unavailable: {exc}')
print('HF_TOKEN=' + ('set' if HF_TOKEN else 'MISSING'))
print(f'RUN_ID={RUN_ID}')
print('comparison_run=20261122_083958; kept separately for a later fair comparison')
print(f'LOG_ROOT={LOG_ROOT}')


In [ ]:
import subprocess
info = subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'], text=True)
print(info.strip())


In [ ]:
import subprocess

def run(command, cwd=None, check=True):
    print('$', ' '.join(str(x) for x in command))
    return subprocess.run(command, cwd=cwd, check=check, text=True)

REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (REPO_DIR / '.git').exists():
    run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_DIR)])
run(['git', 'fetch', '--depth', '1', 'origin', COMMIT], cwd=REPO_DIR)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
if head != COMMIT:
    run(['git', 'reset', '--hard', COMMIT], cwd=REPO_DIR)
else:
    print(f'repo already at {COMMIT}; keep local training patch')
run(['uv', 'sync', '--frozen'], cwd=REPO_DIR)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip())


In [ ]:
import subprocess
from pathlib import Path

motion_dir = REPO_DIR / 'data' / 'lafan1_g1'
motion_dir.mkdir(parents=True, exist_ok=True)
csvs = sorted(motion_dir.glob('*.csv'))
if len(csvs) < 8:
    for item in csvs:
        item.unlink()
    subprocess.run(['bash', 'tools/download_data.sh'], cwd=REPO_DIR, check=True)
    csvs = sorted(motion_dir.glob('*.csv'))
print(f'csv_count={len(csvs)}')
if len(csvs) < 8:
    raise RuntimeError('动作数据不足 8 段')


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

try:
    import huggingface_hub  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)

LOG_ROOT.mkdir(parents=True, exist_ok=True)
need = [name for name in ('args.json', 'last.pt') if not (LOG_ROOT / name).exists()]
if need:
    if not HF_TOKEN:
        raise RuntimeError('缺少本地 checkpoint，且没有 HF_TOKEN 可下载')
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=HF_TOKEN)
    hf_user = api.whoami()['name']
    HF_REPO = f'{hf_user}/{HF_REPO_NAME}'
    print(f'HF_REPO={HF_REPO}')
    tmp = LOG_ROOT / 'hf_download'
    for name in need:
        remote = f'runs/run_{RUN_ID}/{name}'
        path = hf_hub_download(repo_id=HF_REPO, filename=remote, local_dir=str(tmp), token=HF_TOKEN)
        shutil.copy(path, LOG_ROOT / name)
        print(f'downloaded {name}')
else:
    print('checkpoint already local')

print(f'checkpoint={LOG_ROOT / "last.pt"} size={(LOG_ROOT / "last.pt").stat().st_size / 1e6:.1f} MB')


In [ ]:
import json
import os
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor

ckpt = str(LOG_ROOT / 'last.pt')
out_prefix = str(LOG_ROOT / 'eval_final')
traj = LOG_ROOT / 'eval_traj.npz'
record_command = ['uv', 'run', 'python', '-m', 'blinkmimic.evaluate', '--run_dir', str(LOG_ROOT), '--ckpt', ckpt, '--motion_dir', 'data/lafan1_g1', '--record', str(traj), '--device', 'cuda:0']
metrics_command = ['uv', 'run', 'python', 'tools/eval_metrics.py', ckpt, '--out', out_prefix, '--device', 'cuda:0']
def launch(command, device, log_name):
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = device
    env['OMP_NUM_THREADS'] = '4'
    env['MKL_NUM_THREADS'] = '4'
    env['OPENBLAS_NUM_THREADS'] = '4'
    env['NUMEXPR_NUM_THREADS'] = '4'
    start = time.perf_counter()
    result = subprocess.run(command, cwd=REPO_DIR, env=env, text=True, capture_output=True)
    (LOG_ROOT / log_name).write_text(result.stdout + '\n' + result.stderr)
    return result, time.perf_counter() - start
print('parallel rollout: record on GPU 0, full metrics on GPU 1; 4 CPU threads per process')
with ThreadPoolExecutor(max_workers=2) as pool:
    record_future = pool.submit(launch, record_command, '0', 'eval_record_stdout.txt')
    metrics_future = pool.submit(launch, metrics_command, '1', 'eval_metrics_stdout.txt')
    record_result, record_seconds = record_future.result()
    metrics_result, metrics_seconds = metrics_future.result()
print(f'record_seconds={record_seconds:.1f} metrics_seconds={metrics_seconds:.1f}')
print(subprocess.check_output(['nvidia-smi', '--query-gpu=index,memory.used,utilization.gpu', '--format=csv,noheader'], text=True))
for label, result in (('record', record_result), ('metrics', metrics_result)):
    print(f'{label}_exit={result.returncode}')
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError(f'{label} rollout failed')
json_path = out_prefix + '.json'
metrics = json.loads(open(json_path).read())[ckpt]
print(json.dumps(metrics, indent=1))
print(f'traj={traj} size={traj.stat().st_size / 1e6:.1f} MB')


In [ ]:
print(f'eval_rollouts_done metrics={json.dumps(metrics)}')


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

if shutil.which('ffmpeg') is None:
    try:
        import imageio_ffmpeg
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imageio-ffmpeg'], check=True)
        import imageio_ffmpeg
    raw = imageio_ffmpeg.get_ffmpeg_exe()
    bin_dir = Path('/kaggle/working/bin')
    bin_dir.mkdir(parents=True, exist_ok=True)
    link = bin_dir / 'ffmpeg'
    if not link.exists():
        link.symlink_to(raw)
    os.environ['PATH'] = str(bin_dir) + ':' + os.environ['PATH']
    print(f'ffmpeg={raw}')

names = [p.stem for p in sorted((REPO_DIR / 'data' / 'lafan1_g1').glob('*.csv'))]
def pick(fragment):
    return next((n for n in names if fragment in n.lower()), None)

clips = [c for c in (pick('walk'), pick('run'), pick('turn'), pick('dance')) if c]
if len(clips) < 3:
    clips = names[:4]
print('video_clips=', clips)

video = LOG_ROOT / 'eval_video.mp4'
command = [
    'uv', 'run', 'python', 'tools/render.py',
    str(LOG_ROOT / 'eval_traj.npz'), str(video),
    '--clips', *clips,
    '--seconds', str(VIDEO_SECONDS),
    '--fps', str(VIDEO_FPS),
    '--width', str(VIDEO_WIDTH),
    '--height', str(VIDEO_HEIGHT),
    '--title', f'BlinkMimic-T4 {RUN_ID}',
]
env = os.environ.copy()
env['MUJOCO_GL'] = 'egl'
print('$', ' '.join(command))
result = subprocess.run(command, cwd=REPO_DIR, env=env, text=True, capture_output=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
    print('video_render=FAILED')
else:
    print(f'video={video} size={video.stat().st_size / 1e6:.1f} MB')


In [ ]:
from IPython.display import Video, display
from pathlib import Path

video = LOG_ROOT / 'eval_video.mp4'
if video.exists():
    display(Video(str(video), embed=True, width=800))
else:
    print('没有生成视频，指标仍可查看')


In [ ]:
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    hf_user = api.whoami()['name']
    HF_REPO = f'{hf_user}/{HF_REPO_NAME}'
    files = [
        LOG_ROOT / 'eval_final.md',
        LOG_ROOT / 'eval_final.json',
        LOG_ROOT / 'eval_traj.npz',
        LOG_ROOT / 'eval_video.mp4',
        LOG_ROOT / 'eval_stdout.txt',
        LOG_ROOT / 'train_log.csv',
        LOG_ROOT / 'gpu_samples.csv',
    ]
    for path in files:
        if not path.exists():
            continue
        api.upload_file(
            path_or_fileobj=str(path),
            path_in_repo=f'runs/run_{RUN_ID}/eval/{path.name}',
            repo_id=HF_REPO,
            repo_type='model',
        )
        print(f'[hf] uploaded {path.name} ({path.stat().st_size / 1e6:.1f} MB)')
    print(f'HF_REPO={HF_REPO}')
else:
    print('HF_TOKEN 缺失，结果只保留在 Kaggle working')


In [ ]:
import json
from pathlib import Path

meta = json.loads(Path(str(LOG_ROOT / 'eval_final.json')).read_text())
key = str(LOG_ROOT / 'last.pt')
row = meta[key]
print('checkpoint:', key)
for name, value in row.items():
    print(f'{name}={value}')
print(f'train_log={LOG_ROOT / "train_log.csv"}')
print(f'video={LOG_ROOT / "eval_video.mp4"}')
print('scope=simulation_only')
